# 03 — Huấn luyện CNN (S2-CNN)

Gọi `src/train_keras.load_experiment_config` và `src/train_keras.run_training` với một file trong `configs/experiments/`
(E01 baseline, E02 + augmentation). Mô hình nhận ảnh 128×128 RGB pixel 0..255, chuẩn hoá /255 nằm trong mô hình,
class weights `balanced`. Checkpoint, `history.csv`, `config.json` lưu ở Drive `runs/<EXP_ID>_<YYYYMMDD-HHMM>/`.
Phụ trách: Hoàng Bách (dự phòng: Yến).

**Trước khi chạy trên Colab:** Runtime → Change runtime type → **T4 GPU**. Không commit `*.keras`.

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token,
> không `git push`. Muốn sửa logic → sửa `src/` + `tests/` trên nhánh riêng và mở PR.
> Lưu notebook bằng *File → Save a copy in GitHub* vào **nhánh của mình**, xoá output trước khi commit.

## 1. Cài đặt và dữ liệu

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Dữ liệu đã xử lý (tạo bởi notebook 02). Colab: giải nén bản .zip trên Drive nếu chưa có.
from src.common import load_contract
DATA_VERSION = load_contract()["dataset"]["data_version"]      # v1_crop128
PROCESSED = WORK / "processed" / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"
if IN_COLAB and not PROCESSED.exists() and PROCESSED_ZIP.exists():
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)
assert (PROCESSED / "split_manifest.csv").exists(), f"Chưa có {PROCESSED} — chạy notebook 02 trước"
print("PROCESSED:", PROCESSED)

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Chọn thí nghiệm

### Data Audit trước huấn luyện

Audit chỉ đọc dataset `v1_crop128` đã có, kể cả Test để kiểm tra tính toàn vẹn; không đánh giá mô hình.
`CONDITIONAL` / `KEEP_V1_WITH_CONDITIONS` vẫn còn 3 điều kiện chưa xác minh (xem `docs/dataset_audit.md`).
Không chia lại dữ liệu, đổi mapping hoặc ghi đè checkpoint E03.


In [ ]:
from src.data_audit import run_audit

audit = run_audit(PROCESSED, DRIVE / "runs" / "data_audit_v1_crop128")
print(audit["status"], audit["decision"])
print("Pending:", audit["pending_verifications"])
assert audit["automated_checks_passed"], "Dataset integrity errors: inspect audit_summary.json before training"


In [ ]:
# E01/E02: TensorFlow/Keras API. src.train is the E03 PyTorch pipeline.
from src.train_keras import load_experiment_config, model_from_config

EXP_CONFIG = "configs/experiments/E01_baseline.yaml"   # hoặc configs/experiments/E02_augment.yaml
cfg = load_experiment_config(REPO / EXP_CONFIG)
print(json.dumps(cfg, ensure_ascii=False, indent=2))
model_from_config(cfg).summary()

## 3. Huấn luyện

In [ ]:
from src.train_keras import run_training

RUNS = DRIVE / "runs"
RESUME_RUN_DIR = None   # Colab ngắt giữa chừng → đặt RUNS / "E01_<YYYYMMDD-HHMM>" để chạy tiếp từ last.keras
t0 = time.time()
model, history, RUN_DIR = run_training(cfg, PROCESSED, RUNS, run_dir=RESUME_RUN_DIR, resume=RESUME_RUN_DIR is not None)
print(f"{cfg['exp_id']} xong sau {(time.time() - t0) / 60:.1f} phút → {RUN_DIR}")

## 4. Đường cong học tập (Train / Validation)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from src.evaluate import plot_learning_curves

best = min(range(len(history["val_loss"])), key=history["val_loss"].__getitem__)
print(f"Epoch tốt nhất theo val_loss: {best + 1} | val_loss={history['val_loss'][best]:.4f}"
      f" | val_accuracy={history['val_accuracy'][best]:.4f}")
fig, _ = plot_learning_curves(history, title_prefix=cfg["exp_id"])
display(fig); plt.close(fig)
print("Ghi RUN_DIR vào issue S2-CNN và PR; đánh giá bằng notebook 04:", RUN_DIR)